In [1]:
from pathlib import Path


import pandas as pd
import xarray as xr
from imagematerials.util import dataset_to_array
from imagematerials.vehicles.constants import maintenance_lifetime_per_mode

import matplotlib.pyplot as plt

import warnings
from pathlib import Path

import prism

from imagematerials.concepts import knowledge_graph
from imagematerials.factory import ModelFactory
from imagematerials.maintenance import Maintenance
from imagematerials.model import GenericMainModel, GenericMaterials, GenericStocks
from imagematerials.util import export_to_netcdf, import_from_netcdf, rebroadcast_prep_data
from imagematerials.vehicles import (
    preprocess,
)

In [2]:
base_dir = "../data/raw"
prep_fp = Path("prep_vema2.nc")

if not prep_fp.is_file():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        orig_prep_data = preprocess(base_dir)
    export_to_netcdf(orig_prep_data, prep_fp)
prep_data = import_from_netcdf(prep_fp)
share_coords = set()
for cur_type in prep_data["shares"].Type.values:
    share_coords.add(cur_type.split(" - ")[0])
output_coords_type = [x for x in prep_data["stocks"].Type.values if x not in share_coords] + list(prep_data["shares"].coords["Type"].values)
prep_data.pop("shares")
new_prep_data = rebroadcast_prep_data(prep_data, knowledge_graph, dim="Type", output_coords=output_coords_type)
new_prep_data = rebroadcast_prep_data(new_prep_data, knowledge_graph, dim="Region", output_coords=prep_data["stocks"].coords["Region"].values)
new_prep_data["knowledge_graph"] = knowledge_graph

new_prep_data["weights"] = new_prep_data.pop("vehicle_weights")

ValueError: Cannot find relations of Node(name='Bikes', synonyms=[], inherits_from='Vehicles') in input coordinates.

In [ ]:
maintenance_material_pd : pd.DataFrame = pd.read_csv("../data/raw/vehicles/standard_data/all_vehicle_maintenance_image.csv", index_col=0)

maintenance_material_pd['Li'] = 0
maintenance_material_pd['Mn'] = 0
maintenance_material_pd['Ni'] = 0
maintenance_material_pd['Ti'] = 0

stacked_maintenance_material = maintenance_material_pd.set_index("Type").stack().rename_axis(index=["Type", "material"]).reset_index(name="value")

stacked_maintenance_material = stacked_maintenance_material.set_index(["Type", "material"])

stacked_maintenance_material_xr = stacked_maintenance_material.to_xarray()
maintenance_material = dataset_to_array(stacked_maintenance_material_xr, ["Type", "material"], [])

modes = list(maintenance_material.coords['Type'].values)
expected_lifetimes = xr.DataArray(
    data=[maintenance_lifetime_per_mode[mode] for mode in modes],
    dims=["Type"],
    coords={"Type": modes},
    name="vehicle_lifetime"
)

In [ ]:
maintenance_material_pd

In [ ]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [ ]:
# Define the coordinates of all dimensions.
Region = list(prep_data["stocks"].coords["Region"].values)
Time = [t for t in complete_timeline]
Cohort = Time
Type = list(prep_data["stocks"].coords["Type"].values)
material = list(prep_data["material_fractions"].coords["material"].values)

new_prep_data = rebroadcast_prep_data(prep_data, knowledge_graph, dim="Type", output_coords=prep_data["shares"].coords["Type"].values)
new_prep_data = rebroadcast_prep_data(new_prep_data, knowledge_graph, dim="Region", output_coords=prep_data["shares"].coords["Region"].values)
new_prep_data["knowledge_graph"] = knowledge_graph

main_model_factory = ModelFactory(
    new_prep_data, complete_timeline
    ).add(GenericStocks
    ).add(GenericMaterials
    ).add(Maintenance
    ).finish()

In [ ]:
warnings.filterwarnings("ignore")
main_model_factory.simulate(simulation_timeline)

In [ ]:
import matplotlib.pyplot as plt

# Define consistent colors
material_colors = {
    "Steel": "#4B4B4B",
    "Aluminium": "#A9A9A9",
    "Others": "#F0E68C",
    "Plastics": "#1E90FF",
    "Copper": "#B87333",
    "Rubber": "#DC143C",
    "Glass": "#00CED1",
    "Wood": "#8B4513",
    "Fluids": "#FF6347",
    "Lead": "#808080",
    "Neodymium": "#D2691E",  # will be removed after grouping
    "Cobalt": "#0047AB",  # cobalt blue
    "Other Rare Earths": "#FF69B4"  # pink
}

# Helper function to sort columns by total sum
def sort_columns_by_sum(df):
    return df.loc[:, df.sum(axis=0).sort_values(ascending=False).index]

# Helper function for a single train type plot
def plot_material_flow(ax, maint_df, prod_df, title):
    # Filter from 1972
    maint_df = maint_df[maint_df.index >= 2000]
    prod_df = prod_df[prod_df.index >= 2000]

    maint_df = sort_columns_by_sum(maint_df)
    prod_df = sort_columns_by_sum(prod_df)
    
    # Keep only non-zero materials across both
    valid_materials = (
        (maint_df != 0).any(axis=0) |
        (prod_df != 0).any(axis=0)
    )
    maint_df = maint_df.loc[:, valid_materials]
    prod_df = prod_df.loc[:, valid_materials]

    # Consistent material order and colors
    materials = maint_df.columns
    colors = [material_colors.get(mat, "#999999") for mat in materials]

    # Plot maintenance
    maint_df.plot.area(ax=ax, stacked=True, color=colors, alpha=0.85, linewidth=0)
    ax.get_legend().remove()
    maintenance_total = maint_df.sum(axis=1)
    ax.plot(maintenance_total.index, maintenance_total, color="black", linewidth=2, linestyle="--", label="Total Maintenance")

    # Plot production stacked above maintenance
    prod_bottom = maintenance_total.copy()
    for mat in materials:
        top = prod_bottom + prod_df[mat]
        ax.fill_between(
            prod_df.index,
            prod_bottom,
            top,
            #label=f"Production: {mat}",
            color=material_colors.get(mat, "#999999"),
            alpha=0.85,
            #step="mid"
        )
        prod_bottom = top

    # Styling
    ax.set_title(title, fontsize=16)
    ax.set_xlabel("Year", fontsize=14)
    ax.tick_params(labelsize=12)
    ax.grid(visible=True, linestyle="--", alpha=0.3)

        # Add 'maintenance' label in bottom right (white)
    ax.text(
        maint_df.index[-1] - 3,                # rightmost year
        ax.get_ylim()[0] + 0.07 * ax.get_ylim()[1],  # just above bottom
        "Maintenance",
        color="white", fontsize=14, ha="right", va="bottom"
    )

    # Add 'maintenance' label just above the dashed line (black)
    ax.text(
        maint_df.index[-1] - 3,
        maintenance_total.iloc[-1] * 1.1,  # just above last value of line
        "Production",
        color="white", fontsize=14, ha="right", va="bottom"
    )


# === Load data ===
# Maintenance and production for both types
prod_df = main_model_factory.inflow_materials.to_array().sum(dim=["Region", "Type"]).to_pandas()

prod_df = prod_df.rolling(window=5, center=True).mean()

maint_df = main_model_factory.inflow_maintenance.to_array().sum(dim=["Region", "Type"]).to_pandas()

rare_earths = ["Nd", "Mn", "Ni", "Ti","Li"]
maint_df["Other Rare Earths"] = maint_df[rare_earths].sum(axis=1)
prod_df["Other Rare Earths"] = prod_df[rare_earths].sum(axis=1)
maint_df = maint_df.drop(columns=rare_earths, errors="ignore")
prod_df = prod_df.drop(columns=rare_earths, errors="ignore")

rename_map = {
    "Pb": "Lead",
    "Co": "Cobalt",
    "Cu": "Copper"
}
maint_df = maint_df.rename(columns=rename_map)
prod_df = prod_df.rename(columns=rename_map)

# === Create side-by-side plot ===
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

plot_material_flow(axes[0], maint_df, prod_df, "Baseline (cars, vans, busses, trucks)")
plot_material_flow(axes[1], maint_df, prod_df, "With lifetime extension (currently dummy)")

# Shared Y label
axes[0].set_ylabel("Material Flow (kg)", fontsize=14)

# Get all handles
all_handles, all_labels = axes[1].get_legend_handles_labels()

# Separate Total Maintenance
material_handles_labels = [(h, l) for h, l in zip(all_handles, all_labels) if not l.startswith("Total")]
maintenance_handles_labels = [(h, l) for h, l in zip(all_handles, all_labels) if l.startswith("Total")]

# Shared legend (outside)
handles, labels = axes[1].get_legend_handles_labels()
handles = handles[::-1]
labels = labels[::-1]
material_handles_labels = list(zip(*material_handles_labels))
fig.legend(material_handles_labels[0][::-1], material_handles_labels[1][::-1],
           title="Materials", bbox_to_anchor=(0.85, 0.64), loc="center left", fontsize=14, title_fontsize=16)

# Total line legend
if maintenance_handles_labels:
    mh, ml = zip(*maintenance_handles_labels)
    fig.legend(mh, ml, title="Reference Lines", bbox_to_anchor=(0.85, 0.3), loc="center left", fontsize=14, title_fontsize=16)

# Layout optimization for slides
plt.tight_layout()
plt.subplots_adjust(right=0.85)  # Space for legend
plt.show()


In [ ]:
material_colors = {
    "Steel": "#4B4B4B",
    "Aluminium": "#A9A9A9",
    "Others": "#F0E68C",
    "Plastics": "#1E90FF",
    "Copper": "#B87333",
    "Rubber": "#DC143C",
    "Glass": "#00CED1",
    "Wood": "#8B4513",
    "Fluids": "#FF6347",
    "Lead": "#808080",
    "Neodymium": "#D2691E",
    "Cobalt": "#0047AB",
    "Cu": "#B87333"
}

In [ ]:
production_long = main_model_factory.inflow_materials.to_array().sum(dim=["Region", "Type","material"]).to_series()
production_long.name ="Production"
production_df_smooth = production_long.rolling(window=5, center=True).mean()

# Sum maintenance inflow over Region and materials
maintenance_long = main_model_factory.inflow_maintenance.to_array().sum(dim=["Region", "Type","material"]).to_series()
maintenance_long.name = "Maintenance"

both = pd.concat([maintenance_long, production_df_smooth], axis=1).iloc[2:-2]

fig, ax = plt.subplots(figsize=(12, 7))
both.plot.area(ax=ax, stacked=True,)
# Labels and title
ax.set_xlabel("Year")
ax.set_ylabel("Material Flow (Mt)")
ax.set_title("Global Production and Maintenance Material Inflow (bus, truck, car, van)")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), title="Materials")
plt.tight_layout()
plt.show()

In [ ]:
production_df = main_model_factory.inflow_materials.to_array().sum(dim=["Region", "Type"]).to_pandas()

# Sum maintenance inflow over Region and materials
maintenance_df = main_model_factory.inflow_maintenance.to_array().sum(dim=["Region", "Type"]).to_pandas()

# Filter out zero-only materials
nonzero_materials = (
    (maintenance_df != 0).any(axis=0) |
    (production_df != 0).any(axis=0) 
)

maintenance_df = maintenance_df.loc[:, nonzero_materials]
production_df = production_df.loc[:, nonzero_materials]

# Filter from 1972 onwards
maintenance_df = maintenance_df[maintenance_df.index >= 1972]
production_df = production_df[production_df.index >= 1972]

# Total maintenance sum (used for offset and line)
maintenance_total = maintenance_df.sum(axis=1)

production_df_smooth = production_df.rolling(window=5, center=True).mean()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# 1. Ensure consistent material order
all_materials = sorted(set(maintenance_df.columns) | set(production_df.columns))

# 2. Build a consistent colormap
cmap = cm.get_cmap("tab20", len(all_materials))
colors = {mat: cmap(i) for i, mat in enumerate(all_materials)}

# 3. Compute maintenance total
maintenance_total = maintenance_df.sum(axis=1)

# 4. Shift production to sit atop maintenance
production_shifted = production_df.add(maintenance_total, axis=0)

# 5. Plot
fig, ax = plt.subplots(figsize=(12, 7))

# Plot maintenance layer


production_df_smoo = production_df.rolling(window=5, center=True).mean()
s = production_df_smoo.sum().sort_values(ascending = False)

production_df_smoo = production_df_smoo[s.index]

production_df_smoo.plot.area(
    ax=ax,
    stacked=True,
    color=[colors[mat] for mat in maintenance_df.columns],
    alpha=0.7
)

# Black line separating layers
ax.plot(maintenance_total.index, maintenance_total, color="black", linewidth=2.5, label="Maintenance Total")



# Labels and formatting
ax.set_xlabel("Year")
ax.set_ylabel("Material Flow (Mt)")
ax.set_title("Maintenance and Production Material Inflow (cars, vans, trucks, busses)")
ax.set_xlim(maintenance_df.index.min(), maintenance_df.index.max())
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), title="Materials")
plt.tight_layout()
plt.show()


In [ ]:
# Pivot back into wide format with multi-index
from matplotlib import cm


pivot_df = combined_df.pivot_table(index="time", columns=["Source", "Material"], values="Value", aggfunc="sum")

# Separate layers
maintenance_plot = pivot_df["Maintenance"]
production_plot = pivot_df["Production"]

# Use consistent colors
all_materials = sorted(set(maintenance_df.columns) | set(production_df.columns))
cmap = cm.get_cmap("tab20", len(all_materials))
colors = {mat: cmap(i) for i, mat in enumerate(all_materials)}

# Plot
fig, ax = plt.subplots(figsize=(12, 7))

# Maintenance
maintenance_plot.plot.area(
    ax=ax,
    stacked=True,
    color=[colors[mat] for mat in maintenance_plot.columns],
    alpha=0.7
)

# Black line in between
mid_line = maintenance_plot.sum(axis=1)
ax.plot(mid_line.index, mid_line, color="black", linewidth=2.5)

# Production (stacked on top)
production_plot.plot.area(
    ax=ax,
    stacked=True,
    color=[colors[mat] for mat in production_plot.columns],
    alpha=0.7
)

# Final touches
ax.set_xlabel("Year")
ax.set_ylabel("Material Flow (Mt)")
ax.set_title("Combined Material Flow by Source")
ax.set_xlim(pivot_df.index.min(), pivot_df.index.max())
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), title="Materials")
plt.tight_layout()
plt.show()


In [ ]:
# Add a new column to indicate the source
maintenance_long = maintenance_df.copy()
maintenance_long["Source"] = "Maintenance"

production_long = production_df_smooth.copy()
production_long["Source"] = "Production"

# Reset index to bring year into a column
maintenance_long = maintenance_long.reset_index().melt(id_vars=["time", "Source"], var_name="Material", value_name="Value")
production_long = production_long.reset_index().melt(id_vars=["time", "Source"], var_name="Material", value_name="Value")



# Combine into one long DataFrame
combined_df = pd.concat([maintenance_long, production_long], ignore_index=True)

In [ ]:
# Sum general inflow over Region





# Plot
fig, ax = plt.subplots(figsize=(12, 7))

# Plot maintenance materials as stacked area
maintenance_df.plot.area(ax=ax, stacked=True, colormap="tab20", alpha=0.7)

# Plot a thick line on top of maintenance total
ax.plot(maintenance_total.index, maintenance_total, color="black", linewidth=3, label="Total Maintenance")

prod_bottom = maintenance_total.copy()
for material in production_df_smooth.columns:
    top = prod_bottom + production_df_smooth[material]
    ax.fill_between(
        production_df_smooth.index,
        prod_bottom,
        top,
        label=f"Prod: {material}",
        alpha=0.7,
        step="mid"
    )
    prod_bottom = top

# Final touches
ax.set_xlabel("Year")
ax.set_ylabel("Material Flow (Mt)")
ax.set_title("Maintenance and Production Material Flows")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), title="Materials")
plt.tight_layout()
plt.show()



In [ ]:
# Sum over time and material to get total flow per Region × Type


heatmap_data = main_model_factory.inflow_maintenance.to_array().sum(dim=["time","Region"]).to_pandas()

# Plot as heatmap
import seaborn as sns
plt.figure(figsize=(12, 8))
sns.heatmap(heatmap_data, annot=False, cmap="YlGnBu")
plt.title("Total Material Flow by Region and Type")
plt.xlabel("Type")
plt.ylabel("Region")
plt.tight_layout()
plt.show()